# 30. Iterators, generators, and lazy object APIs

**Level:** Intermediate

**Learning outcome:** Distinguish a reusable iterable from a one-pass iterator and avoid materializing unnecessary inventory data.

**Prerequisites:** 12: iteration and container protocols.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](29_Function_Decorators_and_Method_Binding.ipynb) | [Next notebook](31_Attribute_Lookup_Introspection_and_Dynamic_Access.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Mental model
An iterable can provide an iterator through iter(...). An iterator tracks its current position,
returns itself from __iter__, and supplies values through __next__ until it raises StopIteration.
Once exhausted, an ordinary iterator stays exhausted. A reusable collection returns a new iterator
each time, so two report consumers do not steal each other's traversal state.

A generator function contains yield. Calling it creates a generator; execution advances on demand.
A generator is itself a one-pass iterator. Put generation inside __iter__ when every traversal
should get a fresh generator. Return ends a generator normally; do not explicitly raise
StopIteration inside generator code.

Lazy APIs can avoid constructing large intermediate lists, but laziness does not make work free
or ensure resource safety. If iteration owns a live file or session, define when it is closed,
including early termination. The examples below use only in-memory data and do not own resources.
Async iteration is a separate protocol using __aiter__ and __anext__, shown in the async lifecycle
notebook.

## Worked example 1: A one-pass interface iterator

**Problem:** Implement the iterator protocol and prove exhaustion explicitly.

### Step-by-step explanation
1. Capture a stable tuple snapshot and a current index.
2. Return self from __iter__ because this object is the iterator.
3. Raise StopIteration at the end rather than returning None.
4. Consume one item manually, the remainder with list, and then prove the iterator is empty.

**Expected output:**
```text
one-pass iterator exhausted as expected
```

**Common mistake / limitation:** Returning None as an end marker makes None look like a real value and does not terminate a for loop.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [1]:
class InterfaceIterator:
    def __init__(self, names):
        self.names = tuple(names)
        self.position = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.position >= len(self.names):
            raise StopIteration
        value = self.names[self.position]
        self.position += 1
        return value

iterator = InterfaceIterator(["Ethernet1/1", "Ethernet1/2"])
assert iter(iterator) is iterator
assert next(iterator) == "Ethernet1/1"
assert list(iterator) == ["Ethernet1/2"]
assert list(iterator) == []
print("one-pass iterator exhausted as expected")

one-pass iterator exhausted as expected


### Practice 1

Test the same iterator with an empty input tuple.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 2: A reusable collection creates independent iterators

**Problem:** Two report consumers should each see the full device inventory.

### Step-by-step explanation
1. Store data, not one shared iterator.
2. Call iter on the stored data on each __iter__ call.
3. Advance one consumer and show the other has not moved.
4. Start a new traversal to prove collection reuse.

**Expected output:**
```text
independent traversals verified
```

**Common mistake / limitation:** A snapshot of the outer sequence still shares any mutable elements inside it.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [2]:
class Inventory:
    def __init__(self, hostnames):
        self._hostnames = tuple(hostnames)

    def __iter__(self):
        return iter(self._hostnames)

inventory = Inventory(["edge-01", "core-01"])
first, second = iter(inventory), iter(inventory)
assert first is not second
assert next(first) == "edge-01"
assert list(second) == ["edge-01", "core-01"]
assert list(first) == ["core-01"]
assert list(inventory) == ["edge-01", "core-01"]
print("independent traversals verified")

independent traversals verified


### Practice 2

Add a lazy filter method returning a fresh generator of matching hostnames.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 3: A lazy generator method

**Problem:** Produce only the interface labels a caller actually requests.

### Step-by-step explanation
1. Calling names creates a generator but does not yet execute its loop.
2. islice requests only three values from the generator.
3. Use a counter to verify no remaining labels were constructed.
4. Close the partially consumed generator explicitly; no external resource is held here.

**Expected output:**
```text
['Ethernet1/1', 'Ethernet1/2', 'Ethernet1/3'] produced: 3
```

**Common mistake / limitation:** A lazy generator may see changing source state unless its snapshot semantics are defined.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [3]:
from itertools import islice

class InterfacePlan:
    def __init__(self, count):
        if type(count) is not int or count < 0:
            raise ValueError("nonnegative count required")
        self.count = count
        self.produced = 0

    def names(self):
        for number in range(1, self.count + 1):
            self.produced += 1
            yield f"Ethernet1/{number}"

plan = InterfacePlan(10000)
names = plan.names()
assert plan.produced == 0
first_three = list(islice(names, 3))
assert first_three == ["Ethernet1/1", "Ethernet1/2", "Ethernet1/3"]
assert plan.produced == 3
names.close()
print(first_three, "produced:", plan.produced)

['Ethernet1/1', 'Ethernet1/2', 'Ethernet1/3'] produced: 3


### Practice 3

Make the generator yield only odd-numbered interfaces and verify production remains lazy.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Check your understanding: Iterators, generators, and lazy object APIs

**You should now be able to:** Distinguish a reusable iterable from a one-pass iterator and avoid materializing unnecessary inventory data.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/library/stdtypes.html`
- `https://docs.python.org/3.13/reference/datamodel.html`

References describe Python contracts. They do not certify any vendor command or network policy.